# 02 整理热搜话题微博及评论

在收集微博热搜词条后，我们使用爬虫系统，完成了以下工作：

- 使用微博的关键词检索功能，采集与热搜词条相关的微博
- 爬取相关微博信息及其评论数据，并将词条相关信息加入到话题微博信息中
- 收集评论用户的个人信息
- 收集评论用户的历史微博信息，最早时间为2025-01-01

进而得到了以下目录和文件。接下来对这些数据进行整理，并以 Parquet 格式保存。

In [ ]:
TOPIC_WEIBO_PATH = "../data/raw/media_crawler/weibos"

TOPIC_COMMENT_PATH = "../data/raw/media_crawler/comments"

TRENDING_FILE = "../data/raw/weibo_trending/top_1_percent_trendings_detail.txt"

TOPIC_WEIBO_SAVE_PATH = "../data/raw/topic_weibo.parquet"

TOPIC_COMMENT_SAVE_PATH = "../data/raw/topic_comment.parquet"

In [ ]:
import os
import json

topic_weibo_data = []
topic_comment_data = []
trending_data = set()

for content_file in os.listdir(TOPIC_WEIBO_PATH):
    with open(os.path.join(TOPIC_WEIBO_PATH, content_file), "r", encoding="utf-8") as f:
        topic_weibo_data.extend(json.load(f))

for comments_file in os.listdir(TOPIC_COMMENT_PATH):
    with open(os.path.join(TOPIC_COMMENT_PATH, comments_file), "r", encoding="utf-8") as f:
        topic_comment_data.extend(json.load(f))

with open(TRENDING_FILE, "r", encoding="utf-8") as f:
    trending_data.update([line.strip() for line in f if line.strip()])

In [ ]:
from datetime import datetime

trending_data = {}

with open(TRENDING_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        date, name, type, click = line.split(' - ')
        
        date = datetime.fromisoformat(date)
        type = type[1:-1]
        click = int(click.replace(',', ''))

        if name in trending_data:
            existed_trending = trending_data[name]
            existed_date = existed_trending["date"]
            existed_click = existed_trending["click"]

            existed_trending["date"] = min(existed_date,date)
            existed_trending["click"] = max(existed_click, click)
        
        else:
            trending_data[name] = {
                "date": date,
                "type": type,
                "click": click
            }

In [ ]:
import re
from collections import defaultdict
from datetime import datetime
from typing import List, Dict

In [ ]:
def extract_topics(content: str, trending_data) -> str:
    """从内容中提取话题标签,仅返回存在于trending_data中的话题"""
    pattern = r'#[^#]+#'
    topics = re.findall(pattern, content)
    topics = [topic.strip('#') for topic in topics]
    # 只保留存在于trending_set中的话题
    # 需要去掉两侧的#号进行匹配
    filtered_topics = [topic for topic in topics if topic in trending_data]
    return ', '.join(filtered_topics)

In [ ]:
import pandas as pd

weibo_rows = []
# weibo_id_set = set()

for weibo in topic_weibo_data:
        weibo_id = int(weibo.get('note_id'))
        
        # 去除重复微博
        # if weibo_id in weibo_id_set:
        #     continue
        # weibo_id_set.add(weibo_id)

        content = weibo.get('content')
        
        date = weibo.get("create_date_time", "")

        topic = extract_topics(content, trending_data)
        topic = topic if topic else None

        trending_info = trending_data.get(topic, {})
        trending_date = trending_info.get("date", None)
        trending_type = trending_info.get("type", None)
        trending_click = trending_info.get("click", -1)

        weibo_rows.append({
            "weibo_id": weibo_id,
            "content": content,
            "create_time": weibo["create_date_time"],
            "create_time_ts": weibo["create_time"],
            "like_count": int(weibo["liked_count"]),
            "comment_count": int(weibo["comments_count"]),
            "repost_count": int(weibo["shared_count"]),
            # "ip_location": weibo["ip_location"],
            "user_id": int(weibo["user_id"]),
            "screen_name": weibo["nickname"],
            "gender": weibo["gender"],
            "topic": topic,
            "trending_date": trending_date,
            "trending_type": trending_type,
            "trending_click": trending_click
        })

df_topic_weibo = pd.DataFrame(weibo_rows)

# 处理时间字段，将字符串类型转换为 datetime 类型
df_topic_weibo["create_time"] = pd.to_datetime(df_topic_weibo["create_time"])
df_topic_weibo["create_time"] = df_topic_weibo["create_time"].dt.tz_localize(None)

In [ ]:
comment_rows = []
# child_parent_map = {}
comment_id_set = set()

for comment in topic_comment_data:
    
    comment_id = int(comment["comment_id"])
    parent_id = int(comment["parent_comment_id"])

    parent_id = parent_id if parent_id != comment_id else -1

    if comment_id in comment_id_set:
        continue
    comment_id_set.add(comment_id)

    # if comment_id != parent_id:
    #     if parent_id in child_parent_map:
    #         print("存在多级评论")
    #     child_parent_map[comment_id] = parent_id

    date = comment["create_date_time"]
    # 构建评论字典
    try:
        comment_rows.append({
            "comment_id": comment_id,
            "parent_id": parent_id,
            "weibo_id": int(comment["note_id"]),
            "user_id": int(comment["user_id"]),
            "screen_name": comment["nickname"],
            "content": comment["content"],
            "create_time": date,
            "create_time_ts": comment["create_time"],
            "sub_comment_count": int(comment["sub_comment_count"]),
            "like_count": int(comment["comment_like_count"]),
            "ip_location": comment["ip_location"],
            "gender": comment["gender"]
        })
    except Exception as e:
        print(f"处理评论 {comment_id} 时发生错误: {e}")
        exit(0)

df_topic_comment = pd.DataFrame(comment_rows)
# 处理时间字段，将字符串类型转换为 datetime 类型
df_topic_comment["create_time"] = pd.to_datetime(df_topic_comment["create_time"])
df_topic_comment["create_time"] = df_topic_comment["create_time"].dt.tz_localize(None)

In [ ]:
df_topic_weibo.to_parquet(TOPIC_WEIBO_SAVE_PATH, index=False)
df_topic_comment.to_parquet(TOPIC_COMMENT_SAVE_PATH, index=False)

## 整理评论用户信息及微博

In [ ]:
USER_WEIBO_DIR = r'..\data\crawler\weibo_crawler\weibo'

USER_SAVE_PATH = r'..\data\crawler\user_info.parquet'

WEIBO_SAVE_PATH = r'..\data\crawler\user_weibo.parquet'

In [ ]:
import os

user_dirs = [d for d in os.listdir(USER_WEIBO_DIR)
             if os.path.isdir(os.path.join(USER_WEIBO_DIR, d))]

In [ ]:
import os, json
import pandas as pd
from datetime import datetime

user_info_rows = []   # df_user_info
weibo_rows        = []   # df_user_weibo

verified_map = {
    -1: "普通用户", 
    0: "个人认证", 
    1: "政府", 
    2: "企业",
    3: "媒体", 
    4: "校园", 
    5: "网站", 
    6: "应用",
    7: "团体/机构",
    200: "普通用户",  
    220: "个人认证"
}

for uid in user_dirs:
    json_path = os.path.join(USER_WEIBO_DIR, uid, f"{uid}.json")
    if not os.path.exists(json_path):
        continue
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    # ── 1. 用户基本信息 ───────────────────────────────────────
    user_info = data["user"]
    ip_location = user_info["ip_location"].split('（')[0]
    ip_location = ip_location if ip_location else None
    registration_time = user_info["registration_time"]
    registration_time = registration_time if registration_time else None
    verified_type = user_info["verified_type"]
    user_info_rows.append({
        "user_id": int(user_info["id"]), 
        "screen_name": user_info["screen_name"], 
        "gender": user_info["gender"], 
        "ip_location": ip_location, 
        "registration_time": registration_time, 
        "total_weibo_count": user_info["statuses_count"], 
        "follower_count": user_info["followers_count"],
        "following_count": user_info["follow_count"], 
        "description": user_info["description"], 
        "verified": user_info["verified"], 
        "verified_type": verified_type, 
        "verified_type_name": verified_map.get(verified_type, None),
        "user_rank": user_info["urank"]
    })

    # ── 2. 微博列表 ───────────────────────────────────────────
    for weibo in data["weibo"]:
        is_repost = True if "retweet" in weibo else False
        if is_repost:
            repost = weibo["retweet"]

            user_id = repost["user_id"]
            user_id = user_id if user_id else -1

            create_time = repost["created_at"]
            create_time = datetime.fromisoformat(create_time)
            ts = int(create_time.timestamp())

            at_users = repost["at_users"].split(',')
            at_users = ','.join(set(at_users))

            row_repost = {
                "weibo_id": repost["id"], 
                "user_id": user_id, 
                "screen_name": repost["screen_name"],
                "content": repost["text"], 
                "create_time": create_time,
                "create_time_ts": ts,
                # "ip_location": repost["location"] if repost["location"] else "未知", 
                "like_count": repost["attitudes_count"], 
                "comment_count": repost["comments_count"], 
                "repost_count": repost["reposts_count"],
                "topics": repost["topics"], 
                "at_users": at_users, 
                "reposted_weibo_id": -1
            }

        create_time = weibo["created_at"]
        create_time = datetime.fromisoformat(create_time)
        ts = int(create_time.timestamp())

        at_users = weibo["at_users"].split(',')
        at_users = ','.join(set(at_users))

        row = {
            "weibo_id": weibo["id"], 
            "user_id": weibo["user_id"], 
            "screen_name": weibo["screen_name"],
            "content": weibo["text"], 
            "create_time": create_time, 
            "create_time_ts": ts,
            # "ip_location": weibo["location"] if weibo["location"] else "未知", 
            "like_count": weibo["attitudes_count"], 
            "comment_count": weibo["comments_count"], 
            "repost_count": weibo["reposts_count"],
            "topics": weibo["topics"], 
            "at_users": at_users, 
            "reposted_weibo_id": weibo["retweet"]["id"] if is_repost else -1
        }

        weibo_rows.append(row)
        if is_repost:
            weibo_rows.append(row_repost)

# ── 构建 DataFrame ────────────────────────────────────────────────────────
df_user_info  = pd.DataFrame(user_info_rows)
df_user_weibo = pd.DataFrame(weibo_rows)

In [ ]:
df_user_info.to_parquet(USER_SAVE_PATH, index=False)
df_user_weibo.to_parquet(WEIBO_SAVE_PATH, index=False)